# NB-13-MISO · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-13-MISO**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-13-MISO.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This observed MISO example applies the price-component decomposition: common energy cancels between compatible hubs, while congestion and loss differences remain separate.

- [Volume V, Chapter 12 — Nodal price fields and components](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c12)
- [Volume I, §9 — Settlement, fully assembled](https://sherandoresearch.github.io/power-markets-series/vol1_congestion_lmp.html#s9)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-13-MISO', require_key=True)
print(setup_status)


# NB-13-MISO — Energy, congestion, and losses
**Question:** how much of a hub-to-hub LMP difference is congestion, and what
disappears when the common energy term cancels? This observed companion to
NB-13 reuses NB-16-MISO's four hubs for 1–21 September 2026 and adds the matching
system energy series from EWB. No synthetic observations are created.
Prerequisites: nodal price components and signed source-to-sink differences.
We validate source compatibility and the fixed EST clock, reconcile components,
and examine why full LMP differences differ from congestion-only quantities.
Rerunning requires the retained private inputs and repository-local helper.

In [ ]:
from datetime import datetime
import json
import numpy as np
from methods_notebooks.session import Session, setup_plot
session = Session('NB-13-MISO', version='1.43.0')
inputs = session.inputs
nodes = inputs['nodes']
print('Local raw input inspection:', inputs['raw_nodal_rows'][nodes[0]][:1])
reconciled = {name: session.execute('lmp_component_reconciliation', request)['result']['component_reconciliation']
    for name, request in inputs['requests'].items()}
assert all(len(r['rows']) == 1008 and len(r['spreads']) == 504 for r in reconciled.values())
assert all(row['within_tolerance'] for r in reconciled.values() for row in r['rows'])
assert all(row['energy_common'] for r in reconciled.values() for row in r['spreads'])
times = [datetime.fromisoformat(t) for t in inputs['times']]
residuals = {(row['node'], datetime.fromisoformat(row['start'])): row['residual']
    for name in ('ar_il','in_mn') for row in reconciled[name]['rows']}
fields = {'lmp':'total','mcc':'congestion','mlc':'loss'}
path_results = {name: {label: np.array([row[field] for row in r['spreads']])
    for label, field in fields.items()} for name, r in reconciled.items()}
m = dict(audit=inputs['source_audit'], arrays={k:np.asarray(v) for k,v in inputs['arrays'].items()},
    energy=np.asarray(inputs['energy']), times=times,
    residual=np.array([[residuals[node,t] for node in nodes] for t in times]), paths=path_results['ar_il'])
print(json.dumps(m['audit'],indent=2))
print('Hubs:',nodes)

## 1. Join the same hour and source report
The existing 2,016 nodal rows and 504 system rows share the MISO snapshot
`miso-20261014T224032Z-62b47895`. Every joined node/hour has the same source
payload hash as its energy row. All four 504-hour panels are complete and unique;
finite components are required. The helper fails on missing or duplicate keys.

Hour ending 1 begins at 05:00 UTC on the market day and HE24 at 04:00 UTC on the
following day. This is fixed EST, without a daylight-saving shift. These labels
are not joined to PJM calendar conventions or identities. Collector retrieval
after delivery does not establish what an earlier auction participant knew.

The system energy field is derived upstream from the same LMP report's
`LMP − MCC − MLC`; it is not an independent price measurement. Its reported
cross-node deviation is zero, using at least 2,628 nodes per hour. Our four-hub
test therefore checks internal accounting and the join, not independent truth.

In [ ]:
a=m['arrays'];plt=setup_plot();colors=['#88C0D0','#EBCB8B','#A3BE8C','#B48EAD']
fig,axes=plt.subplots(3,1,figsize=(11,8),sharex=True)
axes[0].plot(m['times'],m['energy'],color=colors[0]);axes[0].set(title='Common system energy',ylabel='USD/MWh')
for j,node in enumerate(nodes):
    axes[1].plot(m['times'],a['mcc'][:,j],color=colors[j],label=node,linewidth=.8)
    axes[2].plot(m['times'],a['mlc'][:,j],color=colors[j],linewidth=.8)
axes[1].set(title='Marginal congestion component',ylabel='USD/MWh');axes[1].legend(ncol=2,fontsize=8)
axes[2].set(title='Marginal loss component',ylabel='USD/MWh',xlabel='UTC hour beginning')
for ax in axes:ax.grid(alpha=.3)
fig.autofmt_xdate();fig.tight_layout();plt.show()

## 2. Accounting identity before a model
For every retained node/hour, `LMP = energy + MCC + MLC`. The maximum residual
is approximately 1e-13 USD/MWh, numerical roundoff. The predeclared reconciliation
tolerance is 0.011 USD/MWh for published cent precision; the result is much tighter.
We preserve negative components and do not replace missing entries with zero.
The residual is a diagnostic, not a model forecast error or evidence of alpha.

In [ ]:
np.testing.assert_allclose(a['lmp'],m['energy'][:,None]+a['mcc']+a['mlc'],atol=.011,rtol=0)
for j,node in enumerate(nodes):
    print(node,'max absolute accounting residual:',float(np.max(np.abs(m['residual'][:,j]))))

## 3. Source→sink: the energy term cancels, losses remain
Choose ARKANSAS.HUB→ILLINOIS.HUB with sink minus source orientation.
`ΔLMP = ΔMCC + ΔMLC` because the energy term is common. In this sample the loss
difference spans −9.24 to +33.45 USD/MWh: ignoring it changes the full price spread.
This is a retrospective difference between reported prices, not an independently
executable arbitrage. One MW over one hour converts a price difference to its
numerical USD value, but no volume, trade execution, fees, or funding is assumed.
In particular, the full LMP spread is not asserted to be an FTR payout.

In [ ]:
p=m['paths']
np.testing.assert_allclose(p['lmp']-p['mcc'],p['mlc'],atol=.021,rtol=0)
fig,axes=plt.subplots(2,1,figsize=(11,6),sharex=True)
axes[0].plot(m['times'],p['lmp'],color=colors[0],label='Full LMP difference',linewidth=.9)
axes[0].plot(m['times'],p['mcc'],color=colors[1],label='Congestion difference',linewidth=.8)
axes[0].set(title='ARKANSAS.HUB → ILLINOIS.HUB',ylabel='USD/MWh');axes[0].legend()
axes[1].plot(m['times'],p['mlc'],color=colors[2]);axes[1].axhline(0,color='#D8DEE9',linewidth=.5)
axes[1].set(title='Loss component explains the difference',ylabel='USD/MWh',xlabel='UTC hour beginning')
for ax in axes:ax.grid(alpha=.3)
fig.autofmt_xdate();fig.tight_layout();plt.show()

## 4. Orientation and path composition
Differences derived from one nodal panel reverse sign and compose exactly.
AR→IL plus IL→MN must equal AR→MN, component by component. These identities
need neither a network reconstruction nor independently fitted path models.
They cannot identify physical PTDFs or establish report completeness beyond
this selected panel. Quarterly node-name mapping and broader universe changes
remain separate tasks; no cross-market node mapping is inferred here.

In [ ]:
for name in fields:
    ar_il=path_results['ar_il'][name];il_mn=path_results['il_mn'][name];ar_mn=path_results['ar_mn'][name]
    np.testing.assert_allclose(ar_il+il_mn,ar_mn,atol=1e-12)
    np.testing.assert_allclose(ar_il,-path_results['il_ar'][name],atol=1e-12)
    print(name,'maximum composition residual:',float(np.max(np.abs(ar_il+il_mn-ar_mn))))

## Exercise and next evidence
Why does a successful component check not validate a trading backtest? The
common source makes the identity partly a construction; collector times do
not prove historical availability; and prices do not supply positions, auction
awards, or funding terms. Why is ΔMCC insufficient for the full nodal spread?
Inspect the nonzero ΔMLC plot. A settlement study must first identify the exact
product and its loss, congestion, volume, and funding rules.

Source contracts: retained EWB `miso_da_lmp` and `miso_da_energy_price`
descriptors and matching source-report payloads. These are observed report
values; the companion does not independently re-solve MISO dispatch.

In [ ]:
(session.output/'consumer-summary.json').write_text(json.dumps(dict(
    max_component_residual=float(np.abs(m['residual']).max()),
    paths={k:v.tolist() for k,v in m['paths'].items()},hours=len(m['times']),nodes=nodes),indent=2)+'\n')
summary=session.finish({'observed_only':True,'same_source_payload':True,'fixed_est_validated':True,
    'component_parity':True,'path_composition':True,'new_synthetic_data':False})